In [ ]:
import json
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import Dataset 
from transformers import Trainer, TrainingArguments

In [ ]:
# config
MODEL_NAME = 'openai-community/gpt2'
MAX_LENGTH = 512
OUTPUT_DIR = "./gpt2-instruct"

USER_TOKEN = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"

In [ ]:
data = json.load(open('instruction-data.json'))

def format_input(example):
    instruction = example['instruction']
    input_text = example.get('input', '').strip()
    output = example['output']

    if input_text:
        user_text = f"{instruction}\n\n{input_text}"
    else:
        user_text = instruction

    text = f"{USER_TOKEN}{user_text}{ASSISTANT_TOKEN}{output}"
    return {"text": text}

dataset = Dataset.from_list([format_input(example) for example in data])

In [ ]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
print(len(tokenizer))

In [ ]:
tokenizer.add_special_tokens({
    'pad_token': tokenizer.eos_token,
    'additional_special_tokens': [USER_TOKEN, ASSISTANT_TOKEN]
})

In [ ]:
example = dataset[0]

In [ ]:
def prepare_input(example):
    text = example['text']
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )
    input_ids = enc['input_ids']
    labels = [-100] * len(input_ids)

    assistant_id = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
    pad_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

    start = input_ids.index(assistant_id) + 1
    end = input_ids.index(pad_id) + 1

    for i in range(start, end):
        labels[i] = input_ids[i]

    return {
        'input_ids': input_ids,
        'attention_mask': enc['attention_mask'],
        'labels': labels
    }

In [ ]:
tokenized_ds = dataset.map(prepare_input, remove_columns=['text'])

In [ ]:
model.resize_token_embeddings(len(tokenizer))

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=200,
    lr_scheduler_type='cosine',
    optim='adamw_torch',
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    tokenizer=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)